In [21]:
# =============================================================================
# E-COMMERCE RETENTION ANALYSIS
# Notebook — Data Ingestion & Cleaning
# =============================================================================
# Context: Sales and transaction data from a multi-brand e-commerce store
# selling consumable home cleaning and body care products.
# Goal: Prepare a clean, anonymized dataset for retention analysis.
# Data source: 7 Shopify CSV exports (~208k rows, 81 columns raw)
# =============================================================================

import pandas as pd
import numpy as np
import glob
import hashlib


In [22]:
# =============================================================================
# 1. INGESTION — Merge 7 CSV exports into a single DataFrame
# =============================================================================

df = glob.glob("/Users/mac/Downloads/company_data_project/*.csv")
df = pd.concat([pd.read_csv(f) for f in df], ignore_index=True)

/var/folders/2k/50vdrjc571s98s96tr59jpkm0000gn/T/ipykernel_1228/1866107455.py:6: DtypeWarning: Columns (0: Billing Province, 1: Notes, 2: Employee, 3: Phone, 4: Billing Province Name) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat([pd.read_csv(f) for f in df], ignore_index=True)
/var/folders/2k/50vdrjc571s98s96tr59jpkm0000gn/T/ipykernel_1228/1866107455.py:6: DtypeWarning: Columns (0: Billing Province, 1: Cancelled at, 2: Employee, 3: Billing Province Name) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat([pd.read_csv(f) for f in df], ignore_index=True)
/var/folders/2k/50vdrjc571s98s96tr59jpkm0000gn/T/ipykernel_1228/1866107455.py:6: DtypeWarning: Columns (0: Billing Province, 1: Billing Phone, 2: Shipping Province, 3: Shipping Phone, 4: Cancelled at, 5: Employee, 6: Billing Province Name, 7: Shipping Province Name) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.conc

In [23]:
# =============================================================================
# 2. COLUMN SELECTION — Keep only columns relevant to retention analysis
# =============================================================================
 
columns_to_keep = [
    "Name",             # Order number (human-readable Shopify ID)
    "Email",            # Customer identifier — used to track returning customers
    "Created at",       # Order creation timestamp
    "Paid at",          # Payment confirmation timestamp
    "Financial Status", # paid / pending / refunded / voided
    "Cancelled at",     # Populated only for cancelled orders
    "Subtotal",         # Order value before tax/shipping
    "Total",            # Final order value
    "Discount Code",    # Promo code used (if any)
    "Discount Amount",  # Monetary value of discount applied
    "Refunded Amount",  # Refund amount (if any)
    "Accepts Marketing",# Whether customer opted into marketing
    "Source",           # Acquisition channel
    "Vendor",           # Brand within the merged store
    "Lineitem name",    # Product name
    "Lineitem quantity",# Units ordered per line item
    "Lineitem price",   # Unit price
    "Lineitem sku",     # Product SKU
    "Billing Country",  # Customer country
    "Billing City",     # Customer city
    "Shipping Country", # Delivery country
    "Tags",             # Shopify order tags (e.g. 'returning_customer', categories)
    "Note Attributes",  # Contains first_touch / last_touch attribution data
    "Notes",            # Free-text order notes
    "Payment ID",       # Internal payment reference
]
 
df = df[columns_to_keep]
print(f"Shape after column selection: {df.shape}")
 
 


Shape after column selection: (207979, 25)


In [24]:
# =============================================================================
# 3. INITIAL DATA PROFILING
# =============================================================================
 
print("\n--- Data Types ---")
print(df.dtypes)
 
print("\n--- Missing Values ---")
print(df.isnull().sum().sort_values(ascending=False))
 
print("\n--- Basic Statistics ---")
print(df.describe())
 
 



--- Data Types ---
Name                     str
Email                    str
Created at               str
Paid at                  str
Financial Status         str
Cancelled at             str
Subtotal             float64
Total                float64
Discount Code            str
Discount Amount      float64
Refunded Amount      float64
Accepts Marketing        str
Source                   str
Vendor                   str
Lineitem name            str
Lineitem quantity      int64
Lineitem price       float64
Lineitem sku             str
Billing Country          str
Billing City             str
Shipping Country         str
Tags                     str
Note Attributes          str
Notes                    str
Payment ID               str
dtype: object

--- Missing Values ---
Cancelled at         207911
Notes                207874
Discount Code        194041
Shipping Country     151736
Note Attributes      151168
Billing Country      150957
Billing City         150957
Paid at              

In [25]:
 # =============================================================================
# 4. DTYPE CONVERSION
# Note: done before cleaning to avoid type conflicts downstream
# =============================================================================
 
# Dates
df["Created at"]  = pd.to_datetime(df["Created at"],  utc=True)
df["Paid at"]     = pd.to_datetime(df["Paid at"],     utc=True)
df["Cancelled at"] = pd.to_datetime(df["Cancelled at"], utc=True)



In [26]:
# =============================================================================
# 5. DEDUPLICATION
# Drop fully empty rows and exact duplicate rows
# =============================================================================
 
before = len(df)
df = df.dropna(how="all")
df = df.drop_duplicates()
print(f"\nRows removed (empty + duplicates): {before - len(df)}")
print(f"Shape after deduplication: {df.shape}")


Rows removed (empty + duplicates): 96
Shape after deduplication: (207883, 25)


In [27]:
# =============================================================================
# 6. FORWARD-FILL ORDER-LEVEL COLUMNS
# In Shopify exports, 1 row = 1 line item.
# Order-level metadata (status, totals, dates) only appears on the first
# line item row — this propagates it to all rows of the same order.
# =============================================================================
 
order_cols = [
    "Financial Status", "Paid at", "Accepts Marketing",
    "Subtotal", "Total", "Created at"
]
df[order_cols] = df.groupby("Name")[order_cols].ffill()
 
print(f"\nOrders still missing 'Paid at' after ffill: {df['Paid at'].isna().sum()}")

# # Low-cardinality string columns → category (memory efficient)
for col in ["Financial Status", "Accepts Marketing",
            "Billing Country", "Shipping Country", "Vendor"]:
    df[col] = df[col].astype("category")



Orders still missing 'Paid at' after ffill: 556


In [28]:
# =============================================================================
# 7. VENDOR STANDARDIZATION
# Raw Shopify data has inconsistent brand name formatting
# (mixed case, trailing punctuation, extra spaces)
# =============================================================================
 
df["Vendor"] = (
    df["Vendor"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^\w\s]", "", regex=True)   # remove punctuation
    .str.replace(r"\s+", "_", regex=True)       # spaces to underscores
    .astype("category")
)
 
print("\n--- Vendor distribution after standardization ---")
print(df["Vendor"].value_counts())
print(f"Total distinct vendors: {df['Vendor'].nunique()}")
 
 



--- Vendor distribution after standardization ---
Vendor
wondr                            116217
wondrcare                         37613
powr                              24406
brauzz                            22691
klaaar                             5941
planet_b                            792
yiwu_zunqiang_commodity_coltd       206
Name: count, dtype: int64
Total distinct vendors: 7


In [29]:
# =============================================================================
# 8. MISSING VALUE HANDLING
# Each column is handled based on its business meaning
# =============================================================================
 
# Numeric: no discount/refund means 0, not unknown
df["Discount Amount"]  = df["Discount Amount"].fillna(0)
df["Refunded Amount"]  = df["Refunded Amount"].fillna(0)
 
# Categorical: explicit label for no discount
df["Discount Code"]    = df["Discount Code"].fillna("NO_DISCOUNT")
 
# Free text: empty string prevents downstream string errors
df["Tags"]             = df["Tags"].fillna("")
df["Notes"]            = df["Notes"].fillna("")
df["Note Attributes"]  = df["Note Attributes"].fillna("")
 
print("\n--- Missing values after handling ---")
print(df.isnull().sum().sort_values(ascending=False).head(10))
 
 



--- Missing values after handling ---
Cancelled at        207815
Shipping Country    151640
Billing Country     150861
Billing City        150861
Payment ID          148850
Source              148583
Email                 2369
Paid at                556
Lineitem sku            18
Vendor                  17
dtype: int64


In [30]:
# =============================================================================
# 9. EMAIL ANONYMIZATION (GDPR)
# Email is the primary customer identifier for retention tracking.
# We hash it so it remains consistent across orders while protecting PII.
# SHA-256 is one-way: the original email cannot be recovered.
# =============================================================================
 
def hash_val(val):
    """Hash a value to a 12-character anonymized ID. Blanks stay blank."""
    if pd.isna(val) or str(val).strip() == "":
        return val
    return hashlib.sha256(str(val).encode()).hexdigest()[:12]
 
df["Email"] = df["Email"].apply(hash_val)
 
print(f"\nSample hashed emails:\n{df['Email'].dropna().head(3).tolist()}")
 
 



Sample hashed emails:
['eefffdf4eeaa', 'eefffdf4eeaa', '5776fd6bb70d']


In [31]:
# =============================================================================
# 10. FEATURE ENGINEERING
# =============================================================================
 
# Gross revenue per line item
df["gross_revenue"] = df["Lineitem quantity"] * df["Lineitem price"]
 
# Flag free/gift items — excluded from revenue and purchase count analysis
df["is_gratis"] = df["Lineitem name"].str.contains("gratis", case=False, na=False)
 
# Extract attribution from Note Attributes
df["first_touch_brand"] = df["Note Attributes"].str.extract(
    r"pb_first_touch_brand:\s*([^\n]+)"
)
df["last_touch_brand"] = df["Note Attributes"].str.extract(
    r"pb_last_touch_brand:\s*([^\n]+)"
)
 
# Flag orders with attribution data available
df["has_attribution"] = df["first_touch_brand"].notna()
 
print("\n--- Attribution coverage ---")
print(df[["first_touch_brand", "last_touch_brand"]].notna().mean().rename("coverage"))
 
 



--- Attribution coverage ---
first_touch_brand    0.170278
last_touch_brand     0.146246
Name: coverage, dtype: float64


In [32]:
# =============================================================================
# 11. FINAL VALIDATION
# =============================================================================
 
print("\n--- Final shape ---")
print(df.shape)
 
print("\n--- Date range ---")
print(f"Earliest order: {df['Created at'].min()}")
print(f"Latest order:   {df['Created at'].max()}")
 
print("\n--- Unique counts ---")
print(f"Unique orders:    {df['Name'].nunique()}")
print(f"Unique customers: {df['Email'].nunique()}")
print(f"Unique vendors:   {df['Vendor'].nunique()}")
 
print("\n--- Financial Status breakdown ---")
print(df.drop_duplicates('Name')['Financial Status'].value_counts())
 
print("\n--- Remaining nulls ---")
print(df.isnull().sum().sort_values(ascending=False).head(10))
 



--- Final shape ---
(207883, 30)

--- Date range ---
Earliest order: 2026-01-05 22:45:03+00:00
Latest order:   2026-05-31 11:16:09+00:00

--- Unique counts ---
Unique orders:    59300
Unique customers: 42439
Unique vendors:   7

--- Financial Status breakdown ---
Financial Status
paid                  59001
refunded                230
partially_refunded       64
partially_paid            2
voided                    2
pending                   1
Name: count, dtype: int64

--- Remaining nulls ---
Cancelled at         207815
last_touch_brand     177481
first_touch_brand    172485
Shipping Country     151640
Billing Country      150861
Billing City         150861
Payment ID           148850
Source               148583
Email                  2369
Paid at                 556
dtype: int64


In [33]:
# =============================================================================
# 12. EXPORT CLEAN DATASET
# =============================================================================
 
output_path = "company_data_clean_v1.csv"
df.to_csv(output_path, index=False)
print(f"\nClean dataset saved to: {output_path}")
print(f"Final shape: {df.shape}")




Clean dataset saved to: company_data_clean_v1.csv
Final shape: (207883, 30)
